# 淘宝用户行为数据分析报告
## ——基于 200 万条行为事件的转化漏斗与用户洞察

**分析目标**：识别用户从浏览到购买的转化瓶颈，分析品类流量与转化表现。

**数据范围**：2003017 条用户行为事件（2017-11-25 至 2017-12-03）

---
1.数据质量评估  
2.行为漏斗分析  
3.用户分层分析  
4.品类分析  
5.时间规律  
6.分析报告

### **1.数据质量评估**  
**目标**：确认数据能不能用，哪些字段需要处理。

**指标清单**：

| 方向 | 指标 | 业务含义 |
|------|------|----------|
| 规模 | 行数、列数 | 数据量是否足够 |
| 缺失值 | 各字段缺失数与缺失率 | 哪些字段需要特殊处理 |
| 重复值 | 完全重复行数 | 是否存在重复记录 |
| 唯一值 | 用户数、商品数、品类数 | 数据覆盖范围 |
| 行为分布 | 各类行为的数量与占比 | 漏斗结构是否合理 |
| 时间范围 | 最早/最晚时间 | 数据时间跨度 |
| 异常值 | 时间戳异常记录数 | 是否需要过滤 |

In [1]:
import pandas as pd
import numpy as np

# 加载数据
df = pd.read_csv('UserBehavior.csv',
                 dtype={'user_id': 'int32',
                        'item_id': 'int32',
                        'item_category': 'int32',
                        'behavior_type': 'category'})

print('形状:', df.shape)
print(df.head())

形状: (2003017, 5)
   user_id  item_id  item_category behavior_type   timestamp
0        1  2268318        2520377            pv  1511544070
1      100  2337874        1194311            pv  1511742199
2      100  4492008        4482016            pv  1511766135
3      100   251391        3738615            pv  1511929658
4      100  1203280        4640311            pv  1512123340


#### **代码基础检查**

In [2]:
print('缺失值数量:')
print(df.isnull().sum())
print('\n完全重复行数:')
print(df.duplicated().sum())
print('\n用户唯一值数量:',df['user_id'].nunique())
print('商品类别唯一值数量:',df['item_category'].nunique())
print('商品唯一值数量:',df['item_id'].nunique())

缺失值数量:
user_id          0
item_id          0
item_category    0
behavior_type    0
timestamp        0
dtype: int64

完全重复行数:
0

用户唯一值数量: 712150
商品类别唯一值数量: 7021
商品唯一值数量: 747222


#### **行为分布以及占比**

In [3]:
pd.DataFrame({
    '各类行为数量':df['behavior_type'].value_counts(),
    '各类行为数量占比':df['behavior_type'].value_counts(normalize=True)
})

,各类行为数量,各类行为数量占比
behavior_type,,
pv,1794016,0.895657
cart,111081,0.055457
fav,57668,0.028791
buy,40252,0.020096


#### **过滤时间异常值**

时间戳为 UTC，后续分析统一转换为北京时间（+8 小时）

In [4]:
df = df[(df['timestamp'] >= 1511540000) & (df['timestamp'] <= 1512316000)].copy()
print('过滤后形状:', df.shape)
df['时间'] = pd.to_datetime(df['timestamp'], unit='s')
print('时间范围:', df['时间'].min(), '至', df['时间'].max())
print('\n按日期分布:')
print(df['时间'].dt.date.value_counts().sort_index())

过滤后形状: (1997066, 5)
时间范围: 2017-11-24 16:13:20 至 2017-12-03 15:46:40

按日期分布:
时间
2017-11-24     21624
2017-11-25    210662
2017-11-26    211255
2017-11-27    200401
2017-11-28    197699
2017-11-29    206769
2017-11-30    210298
2017-12-01    223271
2017-12-02    278936
2017-12-03    236151
Name: count, dtype: int64


**数据质量结论**：

数据集原始 2003017 行 × 5 列，无缺失值，无完全重复记录。
覆盖用户 712150 人、商品 747222 个、品类 7021 个。
行为类型包括 pv（浏览）、cart（加购）、fav（收藏）、buy（购买）。

**异常值处理**：
发现少量时间戳异常记录（极小值对应 1905 年，极大值对应 2025 年），
过滤后保留 2017-11-24 至 2017-12-03 的数据，共 1997066 行。

**注意事项**：
- 11-24 和 12-03 是半天数据，日趋势分析时单独标注
- 后续分析基于 1997066 行数据

## **2. 行为漏斗分析**

**目标**：分析用户从浏览到购买的转化路径，找出流失最严重的环节。

**指标清单**：

| 指标 | 业务含义 |
|------|----------|
| 浏览用户数 | 有多少用户产生了兴趣 |
| 收藏用户数 | 有多少用户收藏了商品 |
| 加购用户数 | 有多少用户有购买意向 |
| 购买用户数 | 有多少用户完成了购买 |
| 浏览→加购转化率 | 兴趣转化为购买意向的效率 |
| 浏览→收藏转化率 | 兴趣转化为收藏的效率 |
| 加购→购买转化率 | 购买意向转化为实际购买的效率 |
| 整体转化率 | 从浏览到购买的总体效率 |

#### **各行为去重用户数**

**由于一个用户可能产生多次同样行为，影响实际转化率，所以对各行为用户去重**

In [5]:
pv_user=df[df['behavior_type']=='pv']['user_id'].nunique()
cart_user=df[df['behavior_type']=='cart']['user_id'].nunique()
fav_user=df[df['behavior_type']=='fav']['user_id'].nunique()
buy_user=df[df['behavior_type']=='buy']['user_id'].nunique()

In [6]:
print('浏览去重用户数:',pv_user)
print('加购去重用户数:',cart_user)
print('收藏去重用户数:',fav_user)
print('购买去重用户数:',buy_user)

浏览去重用户数: 676874
加购去重用户数: 96668
收藏去重用户数: 47727
购买去重用户数: 38339


#### **基于浏览转化率**

In [19]:
funnel = pd.DataFrame({
    '环节': ['浏览', '加购', '收藏', '购买'],
    '用户数': [pv_user, cart_user, fav_user, buy_user]
})
funnel['相对浏览转化率'] = (funnel['用户数'] / pv_user * 100).round(2).astype(str) + '%'
funnel['流失人数'] = pv_user - funnel['用户数']
funnel['相邻转化率'] = (funnel['用户数'] / funnel['用户数'].shift(1) * 100).round(2).astype(str) + '%'
funnel.loc[funnel['环节'] == '浏览', '相邻转化率'] = ""
funnel

,环节,用户数,相对浏览转化率,流失人数,相邻转化率
0,浏览,676874,100.0%,0,
1,加购,96668,14.28%,580206,14.28%
2,收藏,47727,7.05%,629147,49.37%
3,购买,38339,5.66%,638535,80.33%


#### **各个环节转化率**

In [17]:
print(f'浏览→加购: {cart_user / pv_user * 100:.2f}%')
print(f'加购→购买: {buy_user / cart_user * 100:.2f}%')
print(f'浏览→收藏: {fav_user / pv_user * 100:.2f}%')
print(f'收藏→购买: {buy_user / fav_user * 100:.2f}%')
print(f'浏览→购买（整体）: {buy_user / pv_user * 100:.2f}%')

浏览→加购: 14.28%
加购→购买: 39.66%
浏览→收藏: 7.05%
收藏→购买: 80.33%


'浏览→购买（整体）: 5.66%'

**行为漏斗分析结论**：

基于 1997066 条行为数据，各环节去重用户数：

| 环节 | 用户数 | 相对浏览转化率 | 流失人数 |
|------|--------|----------------|----------|
| 浏览 | 676874 | 100% | 0 |
| 加购 | 96668 | 14.28% | 580206 |
| 收藏 | 47727 | 7.05% | 629147 |
| 购买 | 38339 | 5.66% | 638535 |

**关键发现**：
- 从浏览到购买，整体转化率仅 5.66%，流失 63.8 万人
- 加购→购买转化率 39.66%，收藏→购买转化率 80.33%
- 收藏用户购买意愿最强，运营可重点唤回“加购未购买”用户

## **3. 用户分层分析**

**目标**：按用户行为组合分层，识别不同价值用户群体，为差异化运营提供依据。

**指标清单**：

| 指标 | 业务含义 |
|------|----------|
| 仅浏览用户数 | 兴趣弱，需要唤醒 |
| 浏览+加购用户数 | 购买意向强，需催付转化 |
| 浏览+收藏用户数 | 观望中，需降价或补货触发 |
| 浏览+购买用户数 | 已转化，需维护复购 |

In [24]:
user_behaviors = df.groupby('user_id')['behavior_type'].apply(set)
def func(x):
    if 'buy' in x:
        return '购买用户'
    elif 'cart' in x:
        return '加购用户'
    elif 'fav' in x:
        return '收藏用户'
    else:
        return '仅浏览用户'
user_type_df=user_behaviors.apply(func).value_counts().reset_index()
user_type_df

,behavior_type,count
0,仅浏览用户,540137
1,加购用户,91578
2,收藏用户,41157
3,购买用户,38339


**用户分层结论**：

基于 711211 个用户的行为组合，按“购买 > 加购 > 收藏 > 仅浏览”的优先级分层：

| 用户类型 | 人数 | 占比 | 业务含义 |
|----------|------|------|----------|
| 仅浏览 | 540137 | 75.94% | 兴趣弱，需唤醒 |
| 加购 | 91578 | 12.88% | 意向强，需催付 |
| 收藏 | 41157 | 5.79% | 观望中，需触发 |
| 购买 | 38339 | 5.39% | 已转化，需维护复购 |

**关键发现**：
- 76% 的用户仅浏览未产生任何互动，是最大的待运营群体
- 加购用户（9.2 万）购买意向最强，是催付转化的核心目标
- 收藏用户转化率最高（80.33%），但人数有限
- 购买用户占比仅 5.39%，整体转化率偏低

## **4. 品类分析**

**目标**：分析各品类的流量与转化表现，找出核心品类、问题品类。

**指标清单**：

| 指标 | 业务含义 |
|------|----------|
| 品类浏览量 | 该品类被多少用户浏览，代表曝光度 |
| 品类购买量 | 该品类被多少用户购买，代表实际销量 |
| 品类转化率 | 购买用户 / 浏览用户，代表转化效率 |

**分析框架**：
- 高浏览 + 低转化 → 问题品类，需优化
- 低浏览 + 高转化 → 潜力品类，可加大流量

#### **统计各商品种类转化率**

In [10]:
cat_summary=pd.DataFrame({
    '浏览量':df[df['behavior_type']=='pv'].groupby('item_category')['user_id'].nunique(),
    '购买量':df[df['behavior_type']=='buy'].groupby('item_category')['user_id'].nunique()
}).fillna(0)
cat_summary=cat_summary[cat_summary['浏览量']>=5]
cat_summary['转化率(%)']=(cat_summary['购买量']/cat_summary['浏览量']*100).round(2)
cat_summary

,浏览量,购买量,转化率(%)
item_category,,,
2171,39.0,1.0,2.56
2410,13.0,0.0,0.00
4907,6.0,1.0,16.67
5064,451.0,8.0,1.77
7769,29.0,0.0,0.00
...,...,...,...
5150761,314.0,4.0,1.27
5155355,5.0,0.0,0.00
5156420,25.0,0.0,0.00


In [26]:
top50 = cat_summary.nlargest(20, '浏览量')
top50=top50.sort_values('转化率(%)', ascending=False).reset_index()
top50

,item_category,浏览量,购买量,转化率(%)
0,2885642,16239.0,586.0,3.61
1,2735466,19816.0,668.0,3.37
2,2640118,12641.0,380.0,3.01
3,1879194,12985.0,272.0,2.09
4,4801426,31987.0,544.0,1.70
5,4217906,16153.0,275.0,1.70
6,3738615,13702.0,193.0,1.41
7,3002561,24292.0,311.0,1.28
8,4145813,50043.0,635.0,1.27
9,149192,17163.0,211.0,1.23


**品类分析结论**：

取浏览量 Top50 品类，按转化率排序后发现：

| 类型 | 特征 | 代表品类 | 策略 |
|------|------|----------|------|
| 问题品类 | 高流量、低转化 | 浏览量 4万+，转化率 0.5% 左右 | 优化详情页、调整价格 |
| 核心品类 | 中低流量、高转化 | 浏览量 5千+，转化率 4%+ | 加大曝光、精准推广 |

**关键发现**：Top50 品类呈“高流量低转化”趋势，流量越大的品类转化率反而越低，可能是日用刚需品，用户比价、详情没有说服力丢失。

**业务建议**：
1. 高流量低转化品类：优化详情页和价格策略，对比竞品价格、详情页做差异化，提升转化效率。
2. 中低流量高转化品类：加大曝光和精准推广，放大优势，吸引更多潜在用户。

## **5. 时间规律分析**

**目标**：用户什么时候最活跃，什么时间段下单最多。

**指标清单**：

| 指标 | 业务含义 |
|------|----------|
| 日期维度 | 用户哪天的浏览、购买、转化率最高 |
| 时段维度 | 用户哪个时间段下单最多 |



#### **提取日期和小时**

In [12]:
df['北京时间'] = df['时间'] + pd.Timedelta(hours=8)
df['日期']=df['北京时间'].dt.date
df['小时_北京'] = df['北京时间'].dt.hour

#### **日期维度**

In [28]:
#按日期统计浏览量、购买量、转化率
day_summary=pd.DataFrame({
    '浏览量':df[df['behavior_type']=='pv'].groupby('日期')['user_id'].nunique(),
    '购买量':df[df['behavior_type']=='buy'].groupby('日期')['user_id'].nunique()
})
day_summary['转化率']=(day_summary['购买量']/day_summary['浏览量']*100).round(2)
day_summary=day_summary.reset_index()
day_summary

,日期,浏览量,购买量,转化率
0,2017-11-25,139662,3968,2.84
1,2017-11-26,143950,4078,2.83
2,2017-11-27,138178,4440,3.21
3,2017-11-28,135685,4175,3.08
4,2017-11-29,140744,4457,3.17
5,2017-11-30,142444,4370,3.07
6,2017-12-01,147155,4035,2.74
7,2017-12-02,186733,5112,2.74
8,2017-12-03,183292,4974,2.71


#### **时段维度**

In [29]:
#把小时拆成4个字段，上午(6-11点)下午(12-17点)晚上(18-23点)深夜(0-5点)
df['时段'] = np.select(
    [df['小时_北京'] < 6, df['小时_北京'] < 12, df['小时_北京'] < 18, df['小时_北京'] >= 18],
    ['深夜', '上午', '下午', '晚上'],
    default='未知'
)
time_summary=pd.DataFrame({
    '浏览量':df[df['behavior_type']=='pv'].groupby('时段')['user_id'].nunique(),
    '购买量':df[df['behavior_type']=='buy'].groupby('时段')['user_id'].nunique()
})
time_summary['转化率']=(time_summary['购买量']/time_summary['浏览量']*100).round(2)
time_summary=time_summary.reset_index()
time_summary

,时段,浏览量,购买量,转化率
0,上午,255405,9181,3.59
1,下午,340439,13826,4.06
2,晚上,414082,14105,3.41
3,深夜,95230,2147,2.25


**时间规律结论**：
当前数据集时间戳为utc时间，转换为北京时间+8小时所以日期维度从11-25开始。

按时间维度和时段维度分析发现：

**日期维度：**   
12 月 2 日开始有大规模流量涌入，浏览量提升约 30%，订单总量同步上涨。11-25至12-3日，转化率在 2.71%-3.21% 之间小幅波动，整体平稳。  
结合时间处于双十二预热期，判断为大促前用户集中浏览比价，流量增加后转化率有所下滑，新增访客购买意愿偏低。

**时段维度：**   
晚上（18-23点）流量最大（41.4万），是用户活跃高峰。  
下午（12-17点）转化率最高（4.06%），用户转化效率高。  
深夜（0-5点）浏览量和转化率均为最低（9.5万、2.25%），非核心时段  

  
**日期维度建议：** 大促前（如 12-02、12-03）：流量增加但转化下降，建议优化详情页、价格策略、优惠券力度，提升新访客转化。  
**时段维度建议：**   
上午：流量和转化均平稳，维持现有投放即可。  
晚上：加大曝光和推送，承接流量高峰。  
下午：集中做转化活动（限时折扣、优惠券）。  
深夜：减少投放，节省预算。

# **6.淘宝用户行为数据分析报告**
## ——基于 200 万条行为事件的转化漏斗与用户洞察

**分析目标**：识别用户从浏览到购买的转化瓶颈，找出高价值用户特征与高转化品类。  
**数据范围**：1997066 条用户行为事件（2017-11-25 至 2017-12-03，北京时间）

---

### **1. 数据质量评估**

数据集原始 2003017 行 × 5 列，无缺失值，无完全重复记录。
覆盖用户 712150 人、商品 747222 个、品类 7021 个。
行为类型包括 pv（浏览）、cart（加购）、fav（收藏）、buy（购买）。

**异常值处理**：
发现少量时间戳异常记录（极小值对应 1905 年，极大值对应 2025 年），
过滤后保留 2017-11-24 至 2017-12-03 的数据，共 1997066 行。

**注意事项**：
- 11-24 和 12-03 是半天数据，日趋势分析时单独标注
- 后续分析基于 1997066 行数据

### **2. 行为漏斗分析**
基于 1997066 条行为数据，各环节去重用户数：

| 环节 | 用户数 | 相对浏览转化率 | 流失人数 |
|------|--------|----------------|----------|
| 浏览 | 676874 | 100% | 0 |
| 加购 | 96668 | 14.28% | 580206 |
| 收藏 | 47727 | 7.05% | 629147 |
| 购买 | 38339 | 5.66% | 638535 |

**关键发现**：
- 从浏览到购买，整体转化率仅 5.66%，流失 63.8 万人
- 加购→购买转化率 39.66%，收藏→购买转化率 80.33%
- 收藏用户购买意愿最强，运营可重点唤回“加购未购买”用户

### **3. 用户分层分析**
基于 711211 个用户的行为组合，按“购买 > 加购 > 收藏 > 仅浏览”的优先级分层：

| 用户类型 | 人数 | 占比 | 业务含义 |
|----------|------|------|----------|
| 仅浏览 | 540137 | 75.94% | 兴趣弱，需唤醒 |
| 加购 | 91578 | 12.88% | 意向强，需催付 |
| 收藏 | 41157 | 5.79% | 观望中，需触发 |
| 购买 | 38339 | 5.39% | 已转化，需维护复购 |

**关键发现**：
- 76% 的用户仅浏览未产生任何互动，是最大的待运营群体
- 加购用户（9.2 万）购买意向最强，是催付转化的核心目标
- 收藏用户转化率最高（80.33%），但人数有限
- 购买用户占比仅 5.39%，整体转化率偏低

### **4. 品类分析**
取浏览量 Top50 品类，按转化率排序后发现：

| 类型 | 特征 | 代表品类 | 策略 |
|------|------|----------|------|
| 问题品类 | 高流量、低转化 | 浏览量 4万+，转化率 0.5% 左右 | 优化详情页、调整价格 |
| 核心品类 | 中低流量、高转化 | 浏览量 5千+，转化率 4%+ | 加大曝光、精准推广 |

**关键发现**：Top50 品类呈“高流量低转化”趋势，流量越大的品类转化率反而越低，可能是日用刚需品，用户比价丢失、详情页没有说服力。

**业务建议**：
1. 高流量低转化品类：优化详情页和价格策略，对比竞品价格、详情页做差异化，提升转化效率。
2. 中低流量高转化品类：加大曝光和精准推广，放大优势，吸引更多潜在用户。

### **5. 时间规律分析**
当前数据集时间戳为utc时间，转换为北京时间+8小时所以日期维度从11-25开始。

**关键发现**：
12-02 起流量大规模涌入（浏览量 +24%），订单同步上涨。11-25 至 12-02 转化率稳定在 2.71%-3.21%。
结合双十二预热期，判断为大促前用户集中浏览比价，新访客购买意愿偏低。  
**建议**：大促前流量增加但转化下降，优化详情页、价格策略、优惠券力度。

### **6. 综合结论**  
**建议：** 
对于用户流失，以及高流量低转化的品类，对商品详情做差异化，调整价格等，提高转化率  
下午转化率最高，继续推出限时折扣、优惠卷等活动，提高转化，晚上流量最大、用户最活跃，扩大商品的曝光、推广，吸引潜在用户，深夜非核心时段，减少流量投入。